Below is example code used to fine-tune the SAM3 pre-trained segmentation base model. The SAM3 base model must be obtained from: https://huggingface.co/facebook/sam3 with your Hugging Face account token. Please provide your own folder names and Hugging Face account token to get started.

In [ ]:
### ⚠️ Google Colab Only Notebook

This notebook is designed to run in **Google Colab**. To start using this notebook, clone this GitHub repo in Colab:
```
!git clone https://github.com/Lynnicia/CryoEM_ultrastructures_top_model_decision_tree/tree/main.git
```
It requires:
- GPU runtime (CUDA) for speed tests
- Linux environment
- Colab-specific install steps (e.g., `!pip install`, `!git clone`, `from google.colab.patches import cv2_imshow`)

❌ Running locally (especially on Windows) may fail due to:
- CUDA/toolchain mismatches
- Missing compiled operators

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install roboflow --no-deps
!pip install pillow
!apt-get update
!apt-get install -y libavif-dev libaom-dev
!pip install --no-cache-dir pillow-avif-plugin pi-heif
!pip install filetype

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="[your_api_key]")
project = rf.workspace("[workspace_name]").project("[project_name]")
version = project.version([version_number])
dataset = version.download("coco-segmentation")

In [ ]:
########################## Define the SAM3 architecture #############################
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
from torchvision import utils
from pycocotools.coco import COCO
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from skimage.measure import regionprops, label



In [ ]:
import os
!git clone https://github.com/facebookresearch/sam3.git

%cd /content/sam3
!git checkout f6e51f5 # February 24, 2026 commit HEAD
!pip install -e "/content/sam3[train,dev]" --break-system-packages -q
for f in os.listdir('/content/sam3/sam3'):
    print(f)
!pip install tidecv
!pip install submitit hydra-core decord

In [ ]:
Token Access to SAM3 model

In [ ]:
%cd /content

!pip install tidecv

!pip install git+https://github.com/huggingface/transformers.git
!pip install -U huggingface_hub
!huggingface-cli login

import os
os.environ["HF_TOKEN"] = "[your_huggingface_token]"

from huggingface_hub import login
login(token="[your_huggingface_token]")


from transformers import Sam3Model, Sam3Processor
model = Sam3Model.from_pretrained(
    "facebook/sam3",
    token=os.environ["HF_TOKEN"]  # not use_auth_token
)
processor = Sam3Processor.from_pretrained("facebook/sam3", use_auth_token=True)

processor = Sam3Processor.from_pretrained(
    "facebook/sam3",
    token=os.environ["HF_TOKEN"]
)

!pip install submitit hydra-core decord

In [ ]:
#write .yaml file  --> M13-v7-redlr-6-saveckpts.yaml
%%writefile /content/sam3/sam3/train/configs/SAM3_data.yaml
# @package _global_
defaults:
  - _self_

# ============================================================================
# Paths Configuration (Chage this to your own paths)
# ============================================================================
paths:
  roboflow_vl_100_root: /content/[roboflow_project_folder_name]
  experiment_log_dir: /content/[roboflow_project_folder_name]/runs
  bpe_path: /content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz # This should be under sam3/assets/bpe_simple_vocab_16e6.txt.gz

# Roboflow dataset configuration
roboflow_train:
  num_images: null # Note: This is the number of images used for training. If null, all images are used.
  supercategory: ${all_roboflow_supercategories.${string:${submitit.job_array.task_index}}}

  # Training transforms pipeline
  train_transforms:
    - _target_: sam3.train.transforms.basic_for_api.ComposeAPI
      transforms:
        - _target_: sam3.train.transforms.filter_query_transforms.FlexibleFilterFindGetQueries
          query_filter:
            _target_: sam3.train.transforms.filter_query_transforms.FilterCrowds
        - _target_: sam3.train.transforms.point_sampling.RandomizeInputBbox
          box_noise_std: 0.1
          box_noise_max: 20
        - _target_: sam3.train.transforms.segmentation.DecodeRle
        - _target_: sam3.train.transforms.basic_for_api.RandomResizeAPI
          sizes:
            _target_: sam3.train.transforms.basic.get_random_resize_scales
            size: ${scratch.resolution}
            min_size: 480
            rounded: false
          max_size:
            _target_: sam3.train.transforms.basic.get_random_resize_max_size
            size: ${scratch.resolution}
          square: true
          consistent_transform: ${scratch.consistent_transform}
        - _target_: sam3.train.transforms.basic_for_api.PadToSizeAPI
          size: ${scratch.resolution}
          consistent_transform: ${scratch.consistent_transform}
        - _target_: sam3.train.transforms.basic_for_api.ToTensorAPI
        - _target_: sam3.train.transforms.filter_query_transforms.FlexibleFilterFindGetQueries
          query_filter:
            _target_: sam3.train.transforms.filter_query_transforms.FilterEmptyTargets
        - _target_: sam3.train.transforms.basic_for_api.NormalizeAPI
          mean: ${scratch.train_norm_mean}
          std: ${scratch.train_norm_std}
        - _target_: sam3.train.transforms.filter_query_transforms.FlexibleFilterFindGetQueries
          query_filter:
            _target_: sam3.train.transforms.filter_query_transforms.FilterEmptyTargets
    - _target_: sam3.train.transforms.filter_query_transforms.FlexibleFilterFindGetQueries
      query_filter:
        _target_: sam3.train.transforms.filter_query_transforms.FilterFindQueriesWithTooManyOut
        max_num_objects: ${scratch.max_ann_per_img}

  # Validation transforms pipeline
  val_transforms:
    - _target_: sam3.train.transforms.basic_for_api.ComposeAPI
      transforms:
        - _target_: sam3.train.transforms.segmentation.DecodeRle
        - _target_: sam3.train.transforms.basic_for_api.RandomResizeAPI
          sizes: ${scratch.resolution}
          max_size:
            _target_: sam3.train.transforms.basic.get_random_resize_max_size
            size: ${scratch.resolution}
          square: true
          consistent_transform: False
        - _target_: sam3.train.transforms.basic_for_api.ToTensorAPI
        - _target_: sam3.train.transforms.basic_for_api.NormalizeAPI
          mean: ${scratch.train_norm_mean}
          std: ${scratch.train_norm_std}

  # loss config (no mask loss)
  # loss:
  #   _target_: sam3.train.loss.sam3_loss.Sam3LossWrapper
  #   matcher: ${scratch.matcher}
  #   o2m_weight: 2.0
  #   o2m_matcher:
  #     _target_: sam3.train.matcher.BinaryOneToManyMatcher
  #     alpha: 0.3
  #     threshold: 0.4
  #     topk: 4
  #   use_o2m_matcher_on_o2m_aux: false # Another option is true
  #   loss_fns_find:
  #     - _target_: sam3.train.loss.loss_fns.Boxes
  #       weight_dict:
  #         loss_bbox: 5.0
  #         loss_giou: 2.0
  #     - _target_: sam3.train.loss.loss_fns.IABCEMdetr
  #       weak_loss: False
  #       weight_dict:
  #         loss_ce: 20.0 # Another option is 100.0
  #         presence_loss: 20.0
  #       pos_weight: 10.0 # Another option is 5.0
  #       alpha: 0.25
  #       gamma: 2
  #       use_presence: True  # Change
  #       pos_focal: false
  #       pad_n_queries: 200
  #       pad_scale_pos: 1.0

  #   loss_fn_semantic_seg: null
  #  scale_by_find_batch_size: ${scratch.scale_by_find_batch_size}


  # NOTE: Loss to be used for training in case of segmentation
  loss:
    _target_: sam3.train.loss.sam3_loss.Sam3LossWrapper
    matcher: ${scratch.matcher}
    o2m_weight: 2.0
    o2m_matcher:
      _target_: sam3.train.matcher.BinaryOneToManyMatcher
      alpha: 0.3
      threshold: 0.4
      topk: 4
    use_o2m_matcher_on_o2m_aux: false
    loss_fns_find:
      - _target_: sam3.train.loss.loss_fns.Boxes
        weight_dict:
          loss_bbox: 5.0
          loss_giou: 2.0
      - _target_: sam3.train.loss.loss_fns.IABCEMdetr
        weak_loss: False
        weight_dict:
          loss_ce: 20.0 # Another option is 100.0
          presence_loss: 20.0
        pos_weight: 10.0 # Another option is 5.0
        alpha: 0.25
        gamma: 2
        use_presence: True  # Change
        pos_focal: false
        pad_n_queries: 200
        pad_scale_pos: 1.0
      - _target_: sam3.train.loss.loss_fns.Masks
        focal_alpha: 0.25
        focal_gamma: 2.0
        weight_dict:
          loss_mask: 200.0
          loss_dice: 10.0
        compute_aux: false
    loss_fn_semantic_seg:
      _target_: sam3.train.loss.loss_fns.SemanticSegCriterion
      presence_head: True
      presence_loss: False  # Change
      focal: True
      focal_alpha: 0.6
      focal_gamma: 2.0
      downsample: False
      weight_dict:
        loss_semantic_seg: 20.0
        loss_semantic_presence: 1.0
        loss_semantic_dice: 30.0
    scale_by_find_batch_size: ${scratch.scale_by_find_batch_size}

# ============================================================================
# Different helper parameters and functions
# ============================================================================
scratch:
  enable_segmentation: True # NOTE: This is the number of queries used for segmentation
  # Model parameters
  d_model: 256
  pos_embed:
    _target_: sam3.model.position_encoding.PositionEmbeddingSine
    num_pos_feats: ${scratch.d_model}
    normalize: true
    scale: null
    temperature: 10000

  # Box processing
  use_presence_eval: True
  original_box_postprocessor:
    _target_: sam3.eval.postprocessors.PostProcessImage
    max_dets_per_img: -1  # infinite detections
    use_original_ids: true
    use_original_sizes_box: true
    use_original_sizes_mask: true
    use_presence: ${scratch.use_presence_eval}
    iou_type: segm

  # Matcher configuration
  matcher:
    _target_: sam3.train.matcher.BinaryHungarianMatcherV2
    focal: true  # with `focal: true` it is equivalent to BinaryFocalHungarianMatcher
    cost_class: 2.0
    cost_bbox: 5.0
    cost_giou: 2.0
    alpha: 0.25
    gamma: 2
    stable: False
  scale_by_find_batch_size: True

  # Image processing parameters
  resolution: 1008  # MUST be 1008, incompatible with 1024
  consistent_transform: False
  max_ann_per_img: 200

# normalization disabled
  train_norm_mean: [0.0, 0.0, 0.0]
  train_norm_std: [1.0, 1.0, 1.0]
# normalization disabled
  val_norm_mean: [0.0, 0.0, 0.0]
  val_norm_std: [1.0, 1.0, 1.0]

  # Training parameters
  num_train_workers: 10
  num_val_workers: 0
  max_data_epochs: 20
  target_epoch_size: 1500
  hybrid_repeats: 1
  context_length: 2
  gather_pred_via_filesys: false

  # Learning rate and scheduler parameters
  lr_scale: 0.4
  lr_transformer: ${times:5e-6,${scratch.lr_scale}}
  lr_vision_backbone: ${times:2.5e-4,${scratch.lr_scale}}
  lr_language_backbone: ${times:5e-5,${scratch.lr_scale}}
  lrd_vision_backbone: 0.9
  wd: 0.1
  scheduler_timescale: 20
  scheduler_warmup: 20
  scheduler_cooldown: 20

  val_batch_size: 4
  collate_fn_val:
    _target_: sam3.train.data.collator.collate_fn_api
    _partial_: true
    repeats: ${scratch.hybrid_repeats}
    dict_key: roboflow100
    with_seg_masks: ${scratch.enable_segmentation} # Note: Set this to true if using segmentation masks!

  gradient_accumulation_steps: 1
  train_batch_size: 4
  collate_fn:
    _target_: sam3.train.data.collator.collate_fn_api
    _partial_: true
    repeats: ${scratch.hybrid_repeats}
    dict_key: all
    with_seg_masks: ${scratch.enable_segmentation} # Note: Set this to true if using segmentation masks!

# ============================================================================
# Trainer Configuration
# ============================================================================

trainer:

  _target_: sam3.train.trainer.Trainer
  skip_saving_ckpts: false
  empty_gpu_mem_cache_after_eval: True
  skip_first_val: True
  max_epochs: ${scratch.max_data_epochs}
  accelerator: cuda
  seed_value: 123
  val_epoch_freq: 10
  mode: train
  gradient_accumulation_steps: ${scratch.gradient_accumulation_steps}

  distributed:
    backend: nccl
    find_unused_parameters: True
    gradient_as_bucket_view: True

  loss:
    all: ${roboflow_train.loss}
    default:
      _target_: sam3.train.loss.sam3_loss.DummyLoss

  data:
    train:
      _target_: sam3.train.data.torch_dataset.TorchDataset
      dataset:
        _target_: sam3.train.data.sam3_image_dataset.Sam3ImageDataset
        limit_ids: ${roboflow_train.num_images}
        transforms: ${roboflow_train.train_transforms}
        load_segmentation: ${scratch.enable_segmentation}
        max_ann_per_img: 500000
        multiplier: 1
        max_train_queries: 50000
        max_val_queries: 50000
        training: true
        use_caching: False
        img_folder: ${paths.roboflow_vl_100_root}/train/
        ann_file: ${paths.roboflow_vl_100_root}/train/_annotations.coco.json

      shuffle: True
      batch_size: ${scratch.train_batch_size}
      num_workers: ${scratch.num_train_workers}
      pin_memory: True
      drop_last: True
      collate_fn: ${scratch.collate_fn}

    val:
      _target_: sam3.train.data.torch_dataset.TorchDataset
      dataset:
        _target_: sam3.train.data.sam3_image_dataset.Sam3ImageDataset
        load_segmentation: ${scratch.enable_segmentation}
        coco_json_loader:
          _target_: sam3.train.data.coco_json_loaders.COCO_FROM_JSON
          include_negatives: true
          category_chunk_size: 2 # Note: You can increase this based on the memory of your GPU.
          _partial_: true
        img_folder: ${paths.roboflow_vl_100_root}/valid/
        ann_file: ${paths.roboflow_vl_100_root}/valid/_annotations.coco.json
        transforms: ${roboflow_train.val_transforms}
        max_ann_per_img: 100000
        multiplier: 1
        training: false

      shuffle: False
      batch_size: ${scratch.val_batch_size}
      num_workers: ${scratch.num_val_workers}
      pin_memory: True
      drop_last: False
      collate_fn: ${scratch.collate_fn_val}


  model:
    _target_: sam3.model_builder.build_sam3_image_model
    bpe_path: ${paths.bpe_path}
    device: cuda
    eval_mode: false
    enable_segmentation: ${scratch.enable_segmentation} # Warning: Enable this if using segmentation.

  meters:
    val:
      roboflow100:
        detection:
          _target_: sam3.eval.coco_writer.PredictionDumper
          iou_type: "segm"
          dump_dir: ${launcher.experiment_log_dir}/dumps/roboflow
          merge_predictions: True
          postprocessor: ${scratch.original_box_postprocessor}
          gather_pred_via_filesys: ${scratch.gather_pred_via_filesys}
          maxdets: 100
          pred_file_evaluators:
            - _target_: sam3.eval.coco_eval_offline.CocoEvaluatorOfflineWithPredFileEvaluators
              gt_path: ${paths.roboflow_vl_100_root}/valid/_annotations.coco.json
              tide: False
              iou_type: "segm"

  optim:
    amp:
      enabled: True
      amp_dtype: bfloat16

    optimizer:
      _target_: torch.optim.AdamW

    gradient_clip:
      _target_: sam3.train.optim.optimizer.GradientClipper
      max_norm: 0.1
      norm_type: 2

    param_group_modifiers:
      - _target_: sam3.train.optim.optimizer.layer_decay_param_modifier
        _partial_: True
        layer_decay_value: ${scratch.lrd_vision_backbone}
        apply_to: 'backbone.vision_backbone.trunk'
        overrides:
          - pattern: '*pos_embed*'
            value: 1.0

    options:
      lr:
        - scheduler:  # transformer and class_embed
            _target_: sam3.train.optim.schedulers.InverseSquareRootParamScheduler
            base_lr: ${scratch.lr_transformer}
            timescale: ${scratch.scheduler_timescale}
            warmup_steps: ${scratch.scheduler_warmup}
            cooldown_steps: ${scratch.scheduler_cooldown}
        - scheduler:
            _target_: sam3.train.optim.schedulers.InverseSquareRootParamScheduler
            base_lr: ${scratch.lr_vision_backbone}
            timescale: ${scratch.scheduler_timescale}
            warmup_steps: ${scratch.scheduler_warmup}
            cooldown_steps: ${scratch.scheduler_cooldown}
          param_names:
            - 'backbone.vision_backbone.*'
        - scheduler:
            _target_: sam3.train.optim.schedulers.InverseSquareRootParamScheduler
            base_lr: ${scratch.lr_language_backbone}
            timescale: ${scratch.scheduler_timescale}
            warmup_steps: ${scratch.scheduler_warmup}
            cooldown_steps: ${scratch.scheduler_cooldown}
          param_names:
            - 'backbone.language_backbone.*'

      weight_decay:
        - scheduler:
            _target_: fvcore.common.param_scheduler.ConstantParamScheduler
            value: ${scratch.wd}
        - scheduler:
            _target_: fvcore.common.param_scheduler.ConstantParamScheduler
            value: 0.0
          param_names:
            - '*bias*'
          module_cls_names: ['torch.nn.LayerNorm']

  checkpoint:
    save_dir: ${launcher.experiment_log_dir}/checkpoints
    save_freq: 0  # 0 only last checkpoint is saved.

  logging:
    tensorboard_writer:
      _target_: sam3.train.utils.logger.make_tensorboard_logger
      log_dir: ${launcher.experiment_log_dir}/tensorboard
      flush_secs: 120
      should_log: True
    wandb_writer: null
    log_dir: ${launcher.experiment_log_dir}/logs
    log_freq: 10

# ============================================================================
# Launcher and Submitit Configuration
# ============================================================================

launcher:
  num_nodes: 1
  gpus_per_node: 2
  experiment_log_dir: ${paths.experiment_log_dir}
  multiprocessing_context: forkserver

submitit:
  account: null
  partition: null
  qos: null
  timeout_hour: 72
  use_cluster: True
  cpus_per_task: 10
  port_range: [10000, 65000]
  constraint: null
  # Uncomment for job array configuration
  job_array:
    num_tasks: 100
    task_index: 0

# ============================================================================
# Available Roboflow Supercategories (for reference)
# ============================================================================

all_roboflow_supercategories:
  - bacteria

In [ ]:
# Fine-tune SAM3 segmentation pre-trained base model
!python sam3/sam3/train/train.py --config '/configs/SAM3_data.yaml' --use-cluster 0 --num-gpus 1

# Set Target Folder

In [ ]:
os.makedirs("[output_folder_name]", exist_ok=True)

TARGET_FOLDER = "[output_folder_name]"

# Validation curves

In [ ]:
# run curves

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc, average_precision_score
from pycocotools.coco import COCO
import json, os
import torch

val_ann_path = "[valid_json_file]"
val_img_folder = "[valid_image_folder]"

with open(val_ann_path) as f:
    val_ann = json.load(f)

cat_name_to_id = {c["name"]: c["id"] for c in val_ann["categories"]}
# Map to 0-indexed for all_true/all_probs: IM=0, OM=1
prompt_to_ch = {"IM": 0, "OM": 1}

# Accumulate pixel-level ground truth and scores across all val images
all_true  = {0: [], 1: []}
all_probs = {0: [], 1: []}

# Build GT mask lookup from COCO annotations
coco_gt = COCO(val_ann_path)

for img_info in val_ann["images"]:
    img_path = os.path.join(val_img_folder, img_info["file_name"])
    pil_image = Image.open(img_path).convert("RGB")
    orig_width, orig_height = pil_image.size

    # ── Ground truth masks per class ──
    gt_masks = {0: np.zeros((orig_height, orig_width), dtype=np.uint8),
                1: np.zeros((orig_height, orig_width), dtype=np.uint8)}

    ann_ids = coco_gt.getAnnIds(imgIds=img_info["id"])
    anns = coco_gt.loadAnns(ann_ids)
    for ann in anns:
        cat_name = cat_name_to_id  # need reverse
        # find category name
        cat_n = next(c["name"] for c in val_ann["categories"] if c["id"] == ann["category_id"])
        if cat_n not in prompt_to_ch:
            continue
        ch = prompt_to_ch[cat_n]
        m = coco_gt.annToMask(ann)
        if m.shape != (orig_height, orig_width):
            m = cv2.resize(m, (orig_width, orig_height), interpolation=cv2.INTER_NEAREST)
        gt_masks[ch] = np.maximum(gt_masks[ch], m)

    # ── SAM3 predictions ──
    with torch.autocast("cuda", dtype=torch.bfloat16):
        with torch.inference_mode():
            dp, ids = make_datapoint(pil_image, ["IM", "OM"])
            dp = transform(dp)
            batch = collate([dp], dict_key="dummy")["dummy"]
            batch = copy_data_to_device(batch, device, non_blocking=True)
            output = model(batch)
            postprocessor_eval = PostProcessImage(
                max_dets_per_img=-1,
                iou_type="segm",
                use_original_sizes_box=True,
                use_original_sizes_mask=True,
                convert_mask_to_rle=False,
                detection_threshold=0.001,
                to_cpu=True,
            )
            results = postprocessor_eval.process_results(output, batch.find_metadatas)

    # Build probability map: max score across all instances per pixel
    for prompt, ch in prompt_to_ch.items():
        prob_map = np.zeros((orig_height, orig_width), dtype=np.float32)
        for mask, score in zip(results[ids[prompt]]["masks"], results[ids[prompt]]["scores"]):
            m = mask.numpy().squeeze().astype(np.float32)
            if m.shape != (orig_height, orig_width):
                m = cv2.resize(m, (orig_width, orig_height), interpolation=cv2.INTER_NEAREST)
            prob_map = np.maximum(prob_map, m * float(score))

        all_true[ch].append(gt_masks[ch].ravel())
        all_probs[ch].append(prob_map.ravel())

    print(f"  Processed: {img_info['file_name']}")

# Concatenate across all images
for ch in [0, 1]:
    all_true[ch]  = np.concatenate(all_true[ch])
    all_probs[ch] = np.concatenate(all_probs[ch])

print("Done collecting predictions. Plotting...")

# ── Your existing plot code (unchanged) ───────────────────────────────────
labels = {0: "IM", 1: "OM"}
classes = [0, 1]

plt.figure(figsize=(7, 5))

for ch in classes:
    y_true = all_true[ch].ravel()
    y_pred = all_probs[ch].ravel()

    if y_true.size == 0:
        print(f"⚠️ Class {ch}: empty arrays, skipping PR curve")
        continue
    unique_vals = np.unique(y_true)
    if len(unique_vals) < 2:
        print(f"⚠️ Class {ch}: only one label present ({unique_vals.tolist()}), skipping PR curve")
        continue

    precision, recall, _ = precision_recall_curve(y_true, y_pred, pos_label=1)
    auprc = average_precision_score(all_true[ch], all_probs[ch])
    plt.plot(recall, precision, linewidth=1.5, label=f"{labels[ch]} {auprc:.3f}")

y_true_all = np.concatenate([all_true[ch].ravel() for ch in classes])
y_pred_all = np.concatenate([all_probs[ch].ravel() for ch in classes])
precision_all, recall_all, _ = precision_recall_curve(y_true_all, y_pred_all)
auprc_all = auc(recall_all, precision_all)
plt.plot(recall_all, precision_all, linestyle="-", linewidth=3, color="blue",
         label=f"all classes {auprc_all:.3f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.grid(False)
plt.ylim(0, 1)
plt.xlim(0, 1)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()

save_path = os.path.join(TARGET_FOLDER, "VALID_Mask_PR_curve.png")
os.makedirs(TARGET_FOLDER, exist_ok=True)
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")


# F1 curve

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(7.69,5))

labels = {0: "IM", 1: "OM"}

# ----- Plot per-class F1 curves -----
for ch in [0, 1]:
    precision, recall, thresholds = precision_recall_curve(
        all_true[ch], all_probs[ch]
    )
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    plt.plot(
        thresholds,
        f1[:-1],  # match threshold length
        label=labels[ch],
        linewidth=1.5
    )

# ----- Compute micro-average across all classes -----
all_true_flat = np.concatenate([all_true[0], all_true[1]])
all_probs_flat = np.concatenate([all_probs[0], all_probs[1]])

precision, recall, thresholds = precision_recall_curve(all_true_flat, all_probs_flat)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

max_idx = np.argmax(f1)
max_f1 = f1[max_idx]
best_conf = thresholds[max_idx]

plt.plot(
    thresholds,
    f1[:-1],
    label=f"all classes {max_f1:.3f} at {best_conf:.3f}",
    linestyle="-",
    color="blue",
    linewidth=3
)

# ----- Plot formatting -----
plt.xlabel("Confidence")
plt.ylabel("F1 score")
plt.title("F1-Confidence Curve")
plt.grid(False)
plt.ylim(0, 1)
plt.xlim(0, 1)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()


save_path = os.path.join(TARGET_FOLDER, "VALID_Mask_F1_curve.png")
os.makedirs(TARGET_FOLDER, exist_ok=True)
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"Saved to {save_path}")

# Validation Metrics

In [ ]:
import json, os
import numpy as np
import torch
import cv2
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils
from sklearn.metrics import precision_recall_curve, average_precision_score
import pandas as pd

val_ann_path = "[valid_json_file]"
val_img_folder = "[valid_image_folder]"

with open(val_ann_path) as f:
    val_ann = json.load(f)

cat_name_to_id = {c["name"]: c["id"] for c in val_ann["categories"]}
prompt_to_ch   = {"IM": 0, "OM": 1}
coco_gt        = COCO(val_ann_path)

# ── Collect predictions ───────────────────────────────────────────────────
predictions  = []   # for COCO instance eval
all_true     = {0: [], 1: []}  # for pixel eval
all_probs    = {0: [], 1: []}

for img_info in val_ann["images"]:
    img_path  = os.path.join(val_img_folder, img_info["file_name"])
    pil_image = Image.open(img_path).convert("RGB")
    orig_width, orig_height = pil_image.size

    # GT masks
    gt = {0: np.zeros((orig_height, orig_width), dtype=np.uint8),
          1: np.zeros((orig_height, orig_width), dtype=np.uint8)}
    for ann in coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=img_info["id"])):
        cat_name = next(c["name"] for c in val_ann["categories"] if c["id"] == ann["category_id"])
        if cat_name not in prompt_to_ch:
            continue
        ch = prompt_to_ch[cat_name]
        m = coco_gt.annToMask(ann)
        if m.shape != (orig_height, orig_width):
            m = cv2.resize(m, (orig_width, orig_height), interpolation=cv2.INTER_NEAREST)
        gt[ch] = np.maximum(gt[ch], m)

    # SAM3 inference
    with torch.autocast("cuda", dtype=torch.bfloat16):
        with torch.no_grad():
            dp, ids = make_datapoint(pil_image, ["IM", "OM"])
            dp = transform(dp)
            batch = collate([dp], dict_key="dummy")["dummy"]
            batch = copy_data_to_device(batch, device, non_blocking=True)
            output = model(batch)
            pp = PostProcessImage(
                max_dets_per_img=-1, iou_type="segm",
                use_original_sizes_box=True, use_original_sizes_mask=True,
                convert_mask_to_rle=False, detection_threshold=0.05, to_cpu=True,
            )
            results = pp.process_results(output, batch.find_metadatas)
    del batch, output
    torch.cuda.empty_cache()

    # Build prob maps + COCO predictions
    for prompt, ch in prompt_to_ch.items():
        cat_id = cat_name_to_id[prompt]
        prob_map = np.zeros((orig_height, orig_width), dtype=np.float32)

        for mask, score in zip(results[ids[prompt]]["masks"], results[ids[prompt]]["scores"]):
            m = mask.numpy().squeeze().astype(np.float32)
            if m.shape != (orig_height, orig_width):
                m = cv2.resize(m, (orig_width, orig_height), interpolation=cv2.INTER_NEAREST)
            prob_map = np.maximum(prob_map, m * float(score))

            # Add to COCO predictions
            binary = (m > 0.5).astype(np.uint8)
            rle = mask_utils.encode(np.asfortranarray(binary))
            rle["counts"] = rle["counts"].decode("utf-8")
            predictions.append({
                "image_id":    img_info["id"],
                "category_id": cat_id,
                "segmentation": rle,
                "score":       float(score),
            })

        all_true[ch].append(gt[ch].ravel())
        all_probs[ch].append(prob_map.ravel())

    print(f"  {img_info['file_name']} done")

# Concatenate pixel arrays
for ch in [0, 1]:
    all_true[ch]  = np.concatenate(all_true[ch])
    all_probs[ch] = np.concatenate(all_probs[ch])

# ── COCO instance eval @IoU50 ─────────────────────────────────────────────
coco_dt   = coco_gt.loadRes(predictions)
coco_eval = COCOeval(coco_gt, coco_dt, "segm")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# Extract per-class AP@IoU50
coco_ap50 = {}
for cat_name, cat_id in cat_name_to_id.items():
    if cat_name not in prompt_to_ch:
        continue
    cat_idx = coco_eval.params.catIds.index(cat_id)
    coco_ap50[cat_name] = round(float(coco_eval.eval["precision"][0, :, cat_idx, 0, 2].mean()), 3)
im_idx  = coco_eval.params.catIds.index(cat_name_to_id["IM"])
om_idx  = coco_eval.params.catIds.index(cat_name_to_id["OM"])
coco_ap50["all"] = round(float(
    np.mean([
        coco_eval.eval["precision"][0, :, im_idx, 0, 2].mean(),
        coco_eval.eval["precision"][0, :, om_idx, 0, 2].mean(),
    ])
), 3)

# ── Pixel metrics @best threshold ────────────────────────────────────────
def best_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return thresholds[np.argmax(f1[:-1])]

def compute_metrics_at_best_f1(all_true, all_probs):
    results = {}
    labels = {0: "IM", 1: "OM"}

    for ch in [0, 1]:
        y_true = all_true[ch]
        y_prob = all_probs[ch]
        thresh = best_threshold(y_true, y_prob)
        y_pred = (y_prob >= thresh).astype(np.uint8)

        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))

        precision = tp / (tp + fp + 1e-8)
        recall    = tp / (tp + fn + 1e-8)
        f1        = 2 * precision * recall / (precision + recall + 1e-8)
        auprc     = average_precision_score(y_true, y_prob)

        results[labels[ch]] = {
            "Mask Precision":    round(precision, 3),
            "Mask Recall":       round(recall, 3),
            "Pixel AUPRC":       round(auprc, 3),
            "Instance mAP50":    coco_ap50[labels[ch]],
            "F1 Score":          round(f1, 3),
            "Best Threshold":    round(float(thresh), 3),
        }

    y_true_all = np.concatenate([all_true[0], all_true[1]])
    y_prob_all  = np.concatenate([all_probs[0], all_probs[1]])
    thresh = best_threshold(y_true_all, y_prob_all)
    y_pred_all = (y_prob_all >= thresh).astype(np.uint8)

    tp = np.sum((y_pred_all == 1) & (y_true_all == 1))
    fp = np.sum((y_pred_all == 1) & (y_true_all == 0))
    fn = np.sum((y_pred_all == 0) & (y_true_all == 1))

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    auprc     = average_precision_score(y_true_all, y_prob_all)

    results["all"] = {
        "Mask Precision":    round(precision, 3),
        "Mask Recall":       round(recall, 3),
        "Pixel AUPRC":       round(auprc, 3),
        "Instance mAP50":    coco_ap50["all"],
        "F1 Score":          round(f1, 3),
        "Best Threshold":    round(float(thresh), 3),
    }

    return pd.DataFrame(results).T

metrics_df = compute_metrics_at_best_f1(all_true, all_probs)
print("\n── SAM3 Validation Metrics ──")
print(metrics_df.to_string())


